In [ ]:
import os
import sys

project_root = "Incremental-Learning-MycroftMinds"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [ ]:
import time

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from river.forest import ARFRegressor
from river.tree import HoeffdingTreeRegressor
from src.bagging import ARFRegressorFODT
from src.trees import fastodt
from src.boosting import IncubationBoost
from src.ensemble_methods import elasticGBDT
from src.evaluation.baseline import PreviousValueRegressor

from src.data.io import load_dataset
from src.evaluation.tuning import hyperparameter_search_tuning
from src.data import dataloader
from src.utils.logging import load_checkpoint_results, list_log_files, view_log_tail

In [ ]:
data_dir = '../../data'

lag_args = {
    'lag_1h':'1H',  # all based on PACF
    'lag_2h':'2H',
    'lag_24h':'24H',
    'lag_168h':'168H' # 24h*7 = 168h, one week ago
    }

# list feature names
columns_to_add_lag = [
            'Global_active_power'
        ]

# target
target_feature = 'Global_active_power'
batch='electricity'
location=None
device_list = None


# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='od_electricity_hour_wfv',  # Prefix for saved files
    time_agg='1h',  # Time aggregation frequency
    new_data=batch,
    location=location
)

In [ ]:
FEATURES = ['time', 'device_id', 'hour', 'weekday', 'month', 'Global_active_power_lag_1h',
            'Global_active_power_lag_2h', 'Global_active_power_24h', 'Global_active_power_lag_168h']

AGGREGATION = 'hourly'  # only affects the name of the aggregation, can be human-readable
DATA_FREQUENCY = 'H'  # 1-Hour frequency
FORECAST_HORIZON = '1h'  # 1-Hour forecast horizon
SPIKE_PROMINENCE = 5  # Prominence for spike detection, adjusted after EDA
USE_RESIDUAL = True

In [ ]:
# Experiment configuration
EXPERIMENT_PREFIX = "1h_wfv"  # This will be used in all result filenames
RESIDUAL_PREFIX = ("resid_" if USE_RESIDUAL else "")  # if we use residuals
BATCH_MODELS_LOG_PREFIX = RESIDUAL_PREFIX + "od_electricity_batch_models"
INCREMENTAL_MODELS_LOG_PREFIX = RESIDUAL_PREFIX + "od_electricity_incremental_models"

print(f"Experiment prefix: {EXPERIMENT_PREFIX}")
print(f"Batch models will be saved to: .../results/{BATCH_MODELS_LOG_PREFIX}/")
print(f"Incremental models will be saved to: .../results/{INCREMENTAL_MODELS_LOG_PREFIX}/")
print(f"Files will be named with pattern: {EXPERIMENT_PREFIX}_ModelName_results.pkl")

### GBDT

In [ ]:
update_freq = '14D'  # Batch learning update frequency

# GBDT hyperparameter tuning
gbdt = GradientBoostingRegressor

param_grid_gbdt = {
    'n_estimators': [25, 50, 100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'max_depth': [3, 7, 12, 15],
    'subsample': [0.6, 0.8, 1.0],
    'random_state': [42]
}

print("Starting GBDT hyperparameter tuning...")

best_params_gbdt = hyperparameter_search_tuning(
    param_grid_gbdt,
    gbdt,
    train,
    valid,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,  # Batch learning update frequency
    search_type='random',
    n_iter=20,
    log_prefix=BATCH_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX,
    save_intermediate=True,
    use_residuals=True
)

In [ ]:
print("GBDT Best Parameters:")
print(best_params_gbdt)

### RandomForest

In [ ]:
update_freq = '14D'  # Batch learning update frequency

# Random Forest hyperparameter tuning
rf = RandomForestRegressor

param_grid_rf = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [3, 5, 7, 10, 15, 20],
    'max_features': [0.3, 0.5, 0.7, 1.0],
    'random_state': [42],
    'n_jobs': [-1]
}

print("Starting Random Forest hyperparameter tuning...")
best_params_rf = hyperparameter_search_tuning(
    param_grid_rf,
    rf,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,  # Batch learning update frequency
    search_type='random',
    n_iter=20,
    log_prefix=BATCH_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

### ARFRegressor

In [ ]:
param_grid = {
    'n_models': [5, 10, 20, 40, 80, 100],
    'max_features': [0.1, 0.3, 0.5, 1.0],
    'max_depth': [7, 10, 30, 50, 100],
    'aggregation_method': ['mean', 'median'],
    'disable_weighted_vote': [True, False],
    'drift_detector': [None],
    'warning_detector': [None],
    'tau': [0.001, 0.005, 0.01, 0.03, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    'grace_period': [0, 10, 24, 48, 96],
}

arf_hf = ARFRegressor

print("Starting ARF (Hoeffding Tree) hyperparameter tuning...")
best_params_arf_hf = hyperparameter_search_tuning(
    param_grid,
    arf_hf,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=20,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

### FastODT

In [ ]:
update_freq = DATA_FREQUENCY#"1h"  # for incremental models put the same as the data frequency to recreate real world scenario

fodt = fastodt.FastODT

param_grid = {
    'depth': [5, 10, 15, 30, 50, 100],
    'grace_period': [0, 10, 24, 48, 96, 168],
    'grace_period_depth': [-1, 0, 1],
    'split_confidence': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5],
    'max_feature': [0.3, 0.5, 0.7, 1.0],
    'tie_threshold': [0.01, 0.05],
    'bin_type': ['adaptive'],
    'max_bins': [8, 16, 32],
    'split_criterion': ['vr_rel']
}

print("Starting FastODT hyperparameter tuning...")
best_params_fastODT = hyperparameter_search_tuning(
    param_grid,
    fodt,
    train,
    valid,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX,
    use_residuals=True
)

### ARF-FastODT

In [ ]:
update_freq = DATA_FREQUENCY#"1h"

arf = ARFRegressorFODT

param_grid = {
    'n_models': [5, 60, 15, 30],
    'max_features': [0.7, 0.3, 0.5],
    'max_depth': [5, 30, 10],
    'aggregation_method': ['mean'],
    'disable_weighted_vote': [False],
    'drift_detector': [None],
    'warning_detector': [None],
    'eagerness': [0],
    'tau': [0.01, 0.05],
    'delta': [0.01, 0.5, 0.05, 0.1, 0.005],
    'grace_period': [0, 24, 48, 96],
    'grace_period_depth': [-1, 0, 1],
    'bin_type': ['mean'],
    'split_criterion': ['vr_rel'],
}

print("Starting ARF (FastODT) hyperparameter tuning...")
best_params_arf = hyperparameter_search_tuning(
    param_grid,
    arf,
    train,
    valid,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX,
    use_residuals=True
)

### HoeffdingTree

In [ ]:
update_freq = "1h"

hoef = HoeffdingTreeRegressor

param_grid = {
    'max_depth': [3, 5, 10, 15, 20],
    'grace_period': [0, 10, 24, 48, 69, 96],
    'tau': [0.001, 0.005, 0.01, 0.03, 0.05, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    #'merit_preprune': [True, False]
}

print("Starting HoeffdingTree hyperparameter tuning...")
best_params_hf = hyperparameter_search_tuning(
    param_grid,
    hoef,
    train,
    valid,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX,
    use_residuals=True
)

### IncubationBoost

In [ ]:
update_freq = "1h"  # for incremental models put the same as the data frequency to recreate real world scenario:

incb = IncubationBoost

param_grid = {

    'length': [5, 10, 20, 30, 40],
    'learning_rate': [0.001, 0.5, 0.01, 0.1],
    'depth': [5, 30, 10, 20],
    'grace_period': [0, 24, 48],
    'grace_period_depth': [-1, 1],
    'split_confidence': [0.01, 0.05, 0.1, 0.5],
    'max_feature': [0.3, 0.5, 0.7, 1.0],
    'tie_threshold': [0.01, 0.05],
    'bin_type': ['adaptive'],
    'max_bins': [8, 16, 32],
    'split_criterion': ['vr_rel'],
    'verbose':[False]
}

print("Starting Incubation Boost hyperparameter tuning...")
best_params_inc = hyperparameter_search_tuning(
    param_grid,
    incb,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=20,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)